# Agentic Medical RAG — LangGraph + vLLM + Hybrid Retrieval

This notebook implements the requested routing architecture:

```text
User query
    ↓
Query Analyser Agent
    ├── Normal Medical QA → difficulty (easy / mid / hard)
    │       └── Hybrid retrieval + MedCPT reranker → answer
    ├── Ambiguous / Contradictory
    │       └── Clearness Agent → clarification (max 10 interactive turns)
    │               └── Relationship + shared Decomposition Agent
    │                       └── Hybrid retrieval + reranker → answer
    ├── Old-Relevant Query
    │       └── Relationship Agent → shared Decomposition Agent
    │               └── Hybrid retrieval + reranker → answer
    ├── Updated-Need Query
    │       └── Tavily trusted-domain search + web scraper → answer
    ├── Privacy Leakage
    │       └── fixed privacy-safe response
    └── Severe ER
            └── immediate emergency escalation response

After answer generation:
- Professional clinical-tone check
- Third-party privacy-leakage check
- Groundedness / valid chunk-ID citation check
- Optional one-pass repair when the answer fails the checks

The notebook follows the uploaded baseline conventions:
- Google Drive support
- MedCPT dense retrieval
- FAISS vector-store format
- Qwen3-8B-AWQ with vLLM
- checkpointed JSONL/CSV/XLSX outputs

It also supports **local-session/local-PC paths** and can optionally build a compatible FAISS vector store if one does not already exist.

## Install dependencies

In [ ]:
import sys, subprocess

packages = [
    "vllm",
    "transformers",
    "sentencepiece",
    "faiss-cpu",
    "rank-bm25",
    "langgraph",
    "tavily-python",
    "beautifulsoup4",
    "requests",
    "pandas",
    "openpyxl",
    "numpy",
    "tqdm",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]
)
print("Dependencies installed.")

## Imports, reproducibility, and runtime

In [ ]:
import os
import re
import gc
import json
import math
import time
import pickle
import hashlib
from pathlib import Path
from typing import TypedDict, Literal, Optional, Any

import numpy as np
import pandas as pd
import torch
import faiss
import requests

from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

IS_COLAB = False
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except Exception:
    pass

print("Colab:", IS_COLAB)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2),
        "GB",
    )

## Configuration

Set `STORAGE_MODE` to:
- `"drive"` for Google Drive paths in Colab
- `"local"` for `/content/...` in Colab or the current working directory on a local PC

The default Drive vector-store filenames match the uploaded baseline notebook.

In [ ]:
# ---------------- USER CONFIGURATION ----------------
STORAGE_MODE = "drive" if IS_COLAB else "local"   # "drive" or "local"

# vLLM generator / agent model
GENERATOR_MODEL = "Qwen/Qwen3-8B-AWQ"
GENERATOR_MAX_TOKENS = 700
VLLM_GPU_MEMORY_UTILIZATION = 0.88
VLLM_MAX_MODEL_LEN = 8192

# MedCPT retrieval stack
QUERY_MODEL_NAME = "ncbi/MedCPT-Query-Encoder"
ARTICLE_MODEL_NAME = "ncbi/MedCPT-Article-Encoder"
RERANKER_MODEL_NAME = "ncbi/MedCPT-Cross-Encoder"

# Adaptive final context sizes requested by the design.
K_RANGES = {
    "easy": (3, 4),
    "mid": (5, 8),
    "hard": (9, 15),
}
DEFAULT_K = {
    "easy": 4,
    "mid": 8,
    "hard": 12,
}

# After independently retrieving each decomposed subquery, merge/deduplicate
# and globally rerank. For multiple subqueries, cap the final evidence set
# to control context growth. A single subquery keeps its own adaptive k.
DECOMPOSED_FINAL_CONTEXT_CAP = 15

# Reciprocal Rank Fusion
RRF_K = 60

# Clarification
MAX_CLARIFICATION_TURNS = 10

# Post-generation repair
AUTO_REPAIR = True
GROUNDEDNESS_REPAIR_THRESHOLD = 0.75

# Current-info web search
TRUSTED_MEDICAL_DOMAINS = [
    "who.int",
    "cdc.gov",
    "fda.gov",
    "nih.gov",
    "ncbi.nlm.nih.gov",
    "nice.org.uk",
    "canada.ca",
]
TAVILY_MAX_RESULTS = 6
WEB_SCRAPE_TOP_N = 4

# Optional vector-store builder.
# Set True only if the FAISS/metadata files are missing and you have a corpus.
BUILD_VECTOR_STORE_IF_MISSING = False
SOURCE_CORPUS_DIR = None  # e.g. "/content/drive/MyDrive/medical_corpus"

# Batch benchmark
RESUME_BATCH = True
MAX_BATCH_ROWS = None  # e.g. 10 for a smoke test, None for all
# ----------------------------------------------------

if IS_COLAB and STORAGE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive")
elif IS_COLAB:
    ROOT = Path("/content/medical_rag_agent")
else:
    ROOT = Path.cwd() / "medical_rag_agent"

ROOT.mkdir(parents=True, exist_ok=True)

# Same default vector-store location used by the baseline notebook.
if STORAGE_MODE == "drive" and IS_COLAB:
    VECTOR_STORE_DIR = Path("/content/drive/MyDrive/vector store")
else:
    VECTOR_STORE_DIR = ROOT / "vector_store"

INDEX_PATH = VECTOR_STORE_DIR / "medquad_medcpt.faiss"
METADATA_PATH = VECTOR_STORE_DIR / "medquad_metadata.pkl"
VECTOR_CONFIG_PATH = VECTOR_STORE_DIR / "medquad_vectorstore_config.json"

OUTPUT_DIR = ROOT / "agentic_medical_rag_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Try the new benchmark first, then fall back to common names.
BENCHMARK_CANDIDATES = [
    ROOT / "Final_Medical_RAG_Benchmark_276_user_prompts.xlsx",
    ROOT / "Final_Medical_RAG_Benchmark_276_updated.xlsx",
    ROOT / "Final_Medical_RAG_Benchmark_276.xlsx",
    ROOT / "medical_rag_200_question_benchmark.xlsx",
]

BATCH_JSONL = OUTPUT_DIR / "agentic_generation_results.jsonl"
BATCH_CSV = OUTPUT_DIR / "agentic_generation_results.csv"
BATCH_XLSX = OUTPUT_DIR / "agentic_generation_results.xlsx"

print("ROOT:", ROOT)
print("Vector store:", VECTOR_STORE_DIR)
print("Output:", OUTPUT_DIR)

## Optional compatible MedCPT FAISS vector-store builder

In [ ]:
def _read_corpus_records(corpus_dir: str | Path):
    """Read .txt, .jsonl, .csv, and .xlsx files into {text, source} records."""
    corpus_dir = Path(corpus_dir)
    if not corpus_dir.exists():
        raise FileNotFoundError(f"Corpus directory not found: {corpus_dir}")

    records = []

    for p in sorted(corpus_dir.rglob("*")):
        if not p.is_file():
            continue

        suffix = p.suffix.lower()

        if suffix == ".txt":
            text = p.read_text(encoding="utf-8", errors="ignore").strip()
            if text:
                records.append({"text": text, "source": str(p)})

        elif suffix == ".jsonl":
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    if not line.strip():
                        continue
                    obj = json.loads(line)
                    text = (
                        obj.get("text")
                        or obj.get("content")
                        or obj.get("passage")
                        or obj.get("document")
                    )
                    if text:
                        records.append({
                            "text": str(text),
                            "source": str(obj.get("source") or p),
                        })

        elif suffix in {".csv", ".xlsx"}:
            df = pd.read_csv(p) if suffix == ".csv" else pd.read_excel(p)
            lower_map = {str(c).lower(): c for c in df.columns}
            text_col = None
            for candidate in ["text", "content", "passage", "document", "answer"]:
                if candidate in lower_map:
                    text_col = lower_map[candidate]
                    break
            if text_col is None:
                continue
            source_col = lower_map.get("source")
            for _, row in df.iterrows():
                text = row.get(text_col)
                if pd.isna(text) or not str(text).strip():
                    continue
                records.append({
                    "text": str(text),
                    "source": str(row.get(source_col)) if source_col else str(p),
                })

    if not records:
        raise ValueError(
            "No readable corpus records found. Supported: txt/jsonl/csv/xlsx "
            "with a text/content/passage/document column."
        )
    return records


def _chunk_words(text: str, chunk_words: int = 280, overlap_words: int = 50):
    words = str(text).split()
    if not words:
        return []
    step = max(1, chunk_words - overlap_words)
    return [
        " ".join(words[i:i + chunk_words])
        for i in range(0, len(words), step)
        if words[i:i + chunk_words]
    ]


@torch.inference_mode()
def build_medcpt_vector_store(
    corpus_dir: str | Path,
    index_path: str | Path,
    metadata_path: str | Path,
    config_path: str | Path,
    batch_size: int = 24,
):
    """Build a cosine-similarity FAISS index compatible with MedCPT query embeddings."""
    raw_records = _read_corpus_records(corpus_dir)

    chunks = []
    for doc_i, rec in enumerate(raw_records):
        for chunk_i, chunk_text in enumerate(_chunk_words(rec["text"])):
            chunks.append({
                "row_id": doc_i,
                "chunk_id": f"doc{doc_i}_chunk{chunk_i}",
                "source": rec["source"],
                "text": chunk_text,
            })

    print("Documents:", len(raw_records), "| chunks:", len(chunks))

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(ARTICLE_MODEL_NAME)
    model = AutoModel.from_pretrained(ARTICLE_MODEL_NAME).to(device)
    model.eval()

    all_vecs = []
    for start in tqdm(range(0, len(chunks), batch_size), desc="Embedding corpus"):
        batch = chunks[start:start + batch_size]
        texts = [x["text"] for x in batch]
        enc = tok(
            texts,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        out = model(**enc)
        emb = out.last_hidden_state[:, 0, :].float().cpu().numpy().astype("float32")
        faiss.normalize_L2(emb)
        all_vecs.append(emb)

    matrix = np.vstack(all_vecs)
    index = faiss.IndexFlatIP(matrix.shape[1])
    index.add(matrix)

    index_path = Path(index_path)
    metadata_path = Path(metadata_path)
    config_path = Path(config_path)
    index_path.parent.mkdir(parents=True, exist_ok=True)

    faiss.write_index(index, str(index_path))
    with open(metadata_path, "wb") as f:
        pickle.dump(chunks, f)

    config = {
        "article_model": ARTICLE_MODEL_NAME,
        "query_model": QUERY_MODEL_NAME,
        "distance": "cosine_via_normalized_inner_product",
        "dimension": int(matrix.shape[1]),
        "chunks": int(len(chunks)),
        "chunk_words": 280,
        "overlap_words": 50,
    }
    config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")

    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Saved:", index_path)
    print("Saved:", metadata_path)
    print("Saved:", config_path)


if (not INDEX_PATH.exists() or not METADATA_PATH.exists()) and BUILD_VECTOR_STORE_IF_MISSING:
    if SOURCE_CORPUS_DIR is None:
        raise ValueError(
            "BUILD_VECTOR_STORE_IF_MISSING=True but SOURCE_CORPUS_DIR is not set."
        )
    build_medcpt_vector_store(
        SOURCE_CORPUS_DIR,
        INDEX_PATH,
        METADATA_PATH,
        VECTOR_CONFIG_PATH,
    )

## Load FAISS metadata and prepare hybrid retrieval

In [ ]:
for p in [INDEX_PATH, METADATA_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing vector-store file: {p}\n"
            "Either point the paths at the existing baseline vector store or "
            "set BUILD_VECTOR_STORE_IF_MISSING=True and SOURCE_CORPUS_DIR."
        )

faiss_index = faiss.read_index(str(INDEX_PATH))
with open(METADATA_PATH, "rb") as f:
    vector_metadata = pickle.load(f)

if faiss_index.ntotal != len(vector_metadata):
    raise ValueError(
        f"FAISS contains {faiss_index.ntotal} vectors but metadata contains "
        f"{len(vector_metadata)} records."
    )

print("FAISS vectors:", faiss_index.ntotal)
print("Dimension:", faiss_index.d)
print("Metadata:", len(vector_metadata))

if VECTOR_CONFIG_PATH.exists():
    print(VECTOR_CONFIG_PATH.read_text(encoding="utf-8")[:2500])

In [ ]:
def lexical_tokens(text: str):
    return re.findall(r"[a-z0-9]+(?:[-'][a-z0-9]+)?", str(text).lower())

bm25_corpus = [lexical_tokens(m.get("text", "")) for m in vector_metadata]
bm25 = BM25Okapi(bm25_corpus)

print("BM25 initialized over", len(bm25_corpus), "chunks.")

## Load MedCPT query encoder + cross-encoder reranker on CPU

They stay on CPU so Qwen3-8B-AWQ can use the Colab GPU through vLLM. This is slower than GPU retrieval but avoids competing with the generator for L4 VRAM.

In [ ]:
RETRIEVAL_DEVICE = "cpu"

query_tokenizer = AutoTokenizer.from_pretrained(QUERY_MODEL_NAME)
query_model = AutoModel.from_pretrained(QUERY_MODEL_NAME).to(RETRIEVAL_DEVICE)
query_model.eval()

rerank_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)
rerank_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME
).to(RETRIEVAL_DEVICE)
rerank_model.eval()

if query_model.config.hidden_size != faiss_index.d:
    raise ValueError(
        f"Query encoder dimension {query_model.config.hidden_size} != "
        f"FAISS dimension {faiss_index.d}"
    )

print("Loaded query encoder:", QUERY_MODEL_NAME)
print("Loaded reranker:", RERANKER_MODEL_NAME)

In [ ]:
@torch.inference_mode()
def embed_query(question: str) -> np.ndarray:
    enc = query_tokenizer(
        [str(question)],
        truncation=True,
        padding=True,
        max_length=64,
        return_tensors="pt",
    )
    enc = {k: v.to(RETRIEVAL_DEVICE) for k, v in enc.items()}
    out = query_model(**enc)
    emb = out.last_hidden_state[:, 0, :].float().cpu().numpy().astype("float32")
    faiss.normalize_L2(emb)
    return emb


def _dense_candidates(query: str, n: int):
    qvec = embed_query(query)
    scores, indices = faiss_index.search(qvec, min(n, faiss_index.ntotal))
    result = []
    for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), 1):
        if idx >= 0:
            result.append((int(idx), rank, float(score)))
    return result


def _bm25_candidates(query: str, n: int):
    scores = np.asarray(bm25.get_scores(lexical_tokens(query)), dtype=float)
    n = min(n, len(scores))
    if n == 0:
        return []
    top = np.argpartition(-scores, n - 1)[:n]
    top = top[np.argsort(-scores[top])]
    return [
        (int(idx), rank, float(scores[idx]))
        for rank, idx in enumerate(top, 1)
    ]


def rrf_fuse(dense, sparse, rrf_k: int = RRF_K):
    fused = {}
    for source_name, rows in [("dense", dense), ("bm25", sparse)]:
        for idx, rank, raw_score in rows:
            item = fused.setdefault(
                idx,
                {
                    "faiss_id": idx,
                    "rrf_score": 0.0,
                    "dense_rank": None,
                    "bm25_rank": None,
                    "dense_score": None,
                    "bm25_score": None,
                },
            )
            item["rrf_score"] += 1.0 / (rrf_k + rank)
            item[f"{source_name}_rank"] = rank
            item[f"{source_name}_score"] = raw_score

    return sorted(fused.values(), key=lambda x: x["rrf_score"], reverse=True)


@torch.inference_mode()
def rerank_candidates(query: str, candidates: list[dict], top_k: int):
    if not candidates:
        return []

    pairs = []
    for c in candidates:
        m = vector_metadata[c["faiss_id"]]
        pairs.append((str(query), str(m.get("text", ""))))

    scores = []
    batch_size = 16
    for start in range(0, len(pairs), batch_size):
        batch = pairs[start:start + batch_size]
        qs = [x[0] for x in batch]
        docs = [x[1] for x in batch]
        enc = rerank_tokenizer(
            qs,
            docs,
            truncation=True,
            padding=True,
            max_length=512,
            return_tensors="pt",
        )
        enc = {k: v.to(RETRIEVAL_DEVICE) for k, v in enc.items()}
        logits = rerank_model(**enc).logits
        if logits.ndim == 2 and logits.shape[1] == 1:
            vals = logits[:, 0]
        else:
            vals = logits.squeeze()
        scores.extend(vals.detach().float().cpu().tolist())

    enriched = []
    for c, score in zip(candidates, scores):
        m = dict(vector_metadata[c["faiss_id"]])
        enriched.append({
            **c,
            "rerank_score": float(score),
            "row_id": m.get("row_id"),
            "chunk_id": m.get("chunk_id"),
            "question": m.get("question"),
            "focus_area": m.get("focus_area"),
            "question_type": m.get("question_type"),
            "source": m.get("source"),
            "text": m.get("text", ""),
        })

    enriched.sort(key=lambda x: x["rerank_score"], reverse=True)
    top = enriched[:top_k]
    for rank, item in enumerate(top, 1):
        item["context_id"] = f"C{rank}"
        item["final_rank"] = rank
    return top


def hybrid_retrieve(query: str, final_k: int):
    t0 = time.perf_counter()
    candidate_k = min(
        faiss_index.ntotal,
        max(24, final_k * 3),
    )

    dense = _dense_candidates(query, candidate_k)
    sparse = _bm25_candidates(query, candidate_k)
    fused = rrf_fuse(dense, sparse)

    # Rerank only the strongest fused pool.
    fused_pool = fused[:min(len(fused), max(30, final_k * 3))]
    top = rerank_candidates(query, fused_pool, top_k=final_k)

    return top, time.perf_counter() - t0


def chunks_to_context(chunks: list[dict]) -> str:
    blocks = []
    for c in chunks:
        header = [
            f"[{c['context_id']}]",
            f"Chunk ID: {c.get('chunk_id')}",
            f"Reranker score: {c.get('rerank_score', 0):.4f}",
        ]
        if c.get("source"):
            header.append(f"Source: {c['source']}")
        if c.get("focus_area"):
            header.append(f"Focus area: {c['focus_area']}")
        header.append("Evidence:")
        header.append(str(c.get("text", "")))
        blocks.append("\n".join(header))
    return "\n\n".join(blocks)

## Load Qwen3-8B in vLLM

In [ ]:
from vllm import LLM, SamplingParams

if not torch.cuda.is_available():
    raise RuntimeError(
        "vLLM generation requires a CUDA GPU for this notebook. "
        "In Colab select a GPU runtime."
    )

agent_llm = LLM(
    model=GENERATOR_MODEL,
    quantization="awq",
    dtype="half",
    trust_remote_code=True,
    tensor_parallel_size=1,
    gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    max_model_len=VLLM_MAX_MODEL_LEN,
    generation_config="vllm",
    seed=SEED,
)

print("Loaded vLLM model:", GENERATOR_MODEL)

In [ ]:
def vllm_chat(
    messages: list[dict],
    max_tokens: int = 500,
    temperature: float = 0.0,
):
    params = SamplingParams(
        temperature=temperature,
        max_tokens=max_tokens,
    )
    t0 = time.perf_counter()
    outputs = agent_llm.chat(
        messages,
        sampling_params=params,
        use_tqdm=False,
        chat_template_kwargs={"enable_thinking": False},
    )
    latency = time.perf_counter() - t0
    req = outputs[0]
    cand = req.outputs[0]
    text = cand.text.strip()
    meta = {
        "latency_s": float(latency),
        "input_tokens": int(len(req.prompt_token_ids or [])),
        "output_tokens": int(len(cand.token_ids or [])),
    }
    return text, meta


def extract_json_object(text: str) -> dict:
    """Best-effort JSON extractor for deterministic local-LLM structured calls."""
    text = str(text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object found in model output:\n{text[:1000]}")
    candidate = match.group(0)

    # Remove common markdown fences/trailing commas conservatively.
    candidate = candidate.replace("```json", "").replace("```", "")
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)

    return json.loads(candidate)

## Query categories and Query Analyser Agent

In [ ]:
QUERY_CATEGORIES = [
    "NORMAL",
    "AMBIGUOUS",
    "CONTRADICTION",
    "OLD_RELEVANT",
    "UPDATED_NEED",
    "PRIVACY_LEAKAGE",
    "SEVERE_ER",
]

ANALYSER_SYSTEM = r"""
You are the routing classifier for a medical RAG system.
Do NOT answer the medical question.

Classify the user's query into exactly ONE primary category:

NORMAL:
Ordinary medical QA answerable from the local medical knowledge base.
Also use NORMAL when the query contains old information that is clearly
unrelated to the current question. Put that distractor in ignored_stale_context.

AMBIGUOUS:
The query is clinically ambiguous and a meaningful answer requires clarification.

CONTRADICTION:
The user's statements materially conflict with each other and clarification is needed.

OLD_RELEVANT:
The user gives older medical history that could materially affect interpretation,
diagnosis, treatment, medication safety, or risk for the current problem.

UPDATED_NEED:
The user explicitly asks for current/recent/latest information, a current guideline,
recent outbreak, current recommendation, or another fact that requires live web evidence.

PRIVACY_LEAKAGE:
The user is asking to obtain or reveal ANOTHER person's private medical/health information.
Do NOT label a query Privacy Leakage merely because the user includes their OWN PHI,
sexual history, ED, STI history, intercourse date, or other sensitive information.

SEVERE_ER:
The prompt itself contains signs that may require immediate emergency assessment,
such as ongoing uncontrolled bleeding, stroke-like deficits, severe breathing difficulty,
possible anaphylaxis, severe chest pain, major trauma, severe overdose, seizure with
persistent impairment, or another clearly time-critical emergency.

PRIORITY when multiple categories overlap:
1) SEVERE_ER if the current user/patient has an emergency.
2) PRIVACY_LEAKAGE if the main request is to reveal another person's private health data.
3) UPDATED_NEED.
4) CONTRADICTION.
5) AMBIGUOUS.
6) OLD_RELEVANT.
7) NORMAL.

For NORMAL, also estimate difficulty:
easy = direct single-fact/simple medical QA
mid = moderate synthesis/comparison or several facts
hard = complex/multi-part/multi-step clinical reasoning

Choose retrieval_k inside:
easy: 3-4
mid: 5-8
hard: 9-15

Return ONLY JSON with:
{
  "category": "...",
  "difficulty": "easy|mid|hard|null",
  "retrieval_k": integer_or_null,
  "clean_retrieval_query": "...",
  "ignored_stale_context": ["..."],
  "needs_clarification": true_or_false,
  "clarification_question": "... or empty",
  "short_reason": "one short audit-friendly reason, no chain-of-thought"
}
"""


def clamp_k(difficulty: str | None, proposed: Any) -> int:
    difficulty = difficulty if difficulty in K_RANGES else "mid"
    lo, hi = K_RANGES[difficulty]
    try:
        k = int(proposed)
    except Exception:
        k = DEFAULT_K[difficulty]
    return max(lo, min(hi, k))


def analyse_query_text(query: str, clarification_context: str = ""):
    content = str(query)
    if clarification_context.strip():
        content += (
            "\n\nAdditional clarification supplied by the user:\n"
            + clarification_context.strip()
        )

    raw, meta = vllm_chat(
        [
            {"role": "system", "content": ANALYSER_SYSTEM},
            {"role": "user", "content": content},
        ],
        max_tokens=450,
    )

    obj = extract_json_object(raw)
    category = str(obj.get("category", "NORMAL")).upper()
    if category not in QUERY_CATEGORIES:
        category = "NORMAL"

    difficulty = obj.get("difficulty")
    if difficulty not in {"easy", "mid", "hard"}:
        difficulty = "mid" if category == "NORMAL" else None

    retrieval_k = (
        clamp_k(difficulty, obj.get("retrieval_k"))
        if category == "NORMAL"
        else None
    )

    return {
        "category": category,
        "difficulty": difficulty,
        "retrieval_k": retrieval_k,
        "clean_retrieval_query": str(
            obj.get("clean_retrieval_query") or query
        ).strip(),
        "ignored_stale_context": obj.get("ignored_stale_context") or [],
        "needs_clarification": bool(obj.get("needs_clarification", False)),
        "clarification_question": str(
            obj.get("clarification_question") or ""
        ).strip(),
        "short_reason": str(obj.get("short_reason") or "").strip(),
    }, meta

## Relationship + shared decomposition agents

In [ ]:
RELATIONSHIP_SYSTEM = r"""
You are a medical query-relationship analyser.
Do not diagnose and do not give a final answer.

Given the user's current problem plus relevant history/clarification, identify only
the clinically plausible relationship(s) that should guide retrieval.
Do not assume causation.

Return ONLY JSON:
{
  "relationship_summary": "2-4 concise sentences",
  "retrieval_focus": ["short concept 1", "short concept 2", "..."]
}
"""

DECOMPOSE_SYSTEM = r"""
You decompose a medical question for evidence retrieval.
Do NOT answer it.

Create the MINIMUM number of self-contained retrieval subqueries needed to answer
the user's question. Usually 1-3; use at most 4.
Preserve clinically relevant history and remove irrelevant identifiers/distractors.

Return ONLY JSON:
{
  "subqueries": ["...", "..."]
}
"""


def relationship_analysis(query: str, clarification_context: str = ""):
    raw, meta = vllm_chat(
        [
            {"role": "system", "content": RELATIONSHIP_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Original query:\n{query}\n\n"
                    f"Clarification/history context:\n{clarification_context}"
                ),
            },
        ],
        max_tokens=350,
    )
    obj = extract_json_object(raw)
    return {
        "relationship_summary": str(obj.get("relationship_summary") or "").strip(),
        "retrieval_focus": obj.get("retrieval_focus") or [],
    }, meta


def decompose_query(
    query: str,
    relationship_summary: str = "",
    clarification_context: str = "",
):
    raw, meta = vllm_chat(
        [
            {"role": "system", "content": DECOMPOSE_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Question:\n{query}\n\n"
                    f"Relationship summary:\n{relationship_summary}\n\n"
                    f"Additional clarification:\n{clarification_context}"
                ),
            },
        ],
        max_tokens=350,
    )
    obj = extract_json_object(raw)
    subqueries = [
        str(x).strip()
        for x in (obj.get("subqueries") or [])
        if str(x).strip()
    ]
    if not subqueries:
        subqueries = [query]
    return subqueries[:4], meta

SUBQUERY_DIFFICULTY_SYSTEM = r"""
You are a medical retrieval-difficulty classifier.
Do NOT answer the question.

Classify ONE decomposed medical retrieval subquery as:
easy = direct single-fact/simple medical QA
mid = moderate synthesis/comparison or several facts
hard = complex/multi-step/high-risk clinical reasoning

Choose retrieval_k inside:
easy: 3-4
mid: 5-8
hard: 9-15

Return ONLY JSON:
{
  "difficulty": "easy|mid|hard",
  "retrieval_k": integer,
  "short_reason": "one concise audit note"
}
"""


def classify_subquery_difficulty(subquery: str):
    raw, meta = vllm_chat(
        [
            {"role": "system", "content": SUBQUERY_DIFFICULTY_SYSTEM},
            {"role": "user", "content": str(subquery)},
        ],
        max_tokens=220,
    )
    obj = extract_json_object(raw)

    difficulty = str(obj.get("difficulty", "mid")).lower()
    if difficulty not in {"easy", "mid", "hard"}:
        difficulty = "mid"

    retrieval_k = clamp_k(difficulty, obj.get("retrieval_k"))

    return {
        "subquery": str(subquery).strip(),
        "difficulty": difficulty,
        "retrieval_k": retrieval_k,
        "short_reason": str(obj.get("short_reason") or "").strip(),
    }, meta


def build_subquery_plans(subqueries: list[str]):
    plans = []
    metas = []
    for sq in subqueries:
        plan, meta = classify_subquery_difficulty(sq)
        plans.append(plan)
        metas.append(meta)
    return plans, metas


## RAG generation with chunk-ID citations

In [ ]:
RAG_SYSTEM = r"""
You are an evidence-grounded medical question-answering assistant.

Rules:
1. Base the medical answer on the supplied retrieved evidence.
2. Cite supporting retrieved chunks inline using their IDs exactly, e.g. [C1], [C3].
3. Never invent a chunk ID.
4. If the evidence is insufficient, say so explicitly instead of guessing.
5. Use professional clinical language, but do not claim to be a physician.
6. Be clear about uncertainty.
7. Do not reveal private health information belonging to another user/person.
8. Minimize repetition of the user's own identifying information unless clinically needed.
9. If the evidence indicates urgent/emergency care is warranted, say so plainly.
"""


def _retrieve_planned_subqueries(
    original_query: str,
    subquery_plans: list[dict],
):
    """
    Retrieve EACH decomposed subquery with its own adaptive difficulty-based k:
      easy -> 3-4
      mid  -> 5-8
      hard -> 9-15

    Then merge/deduplicate and globally rerank against the original query.
    """
    all_chunks = {}
    total_latency = 0.0
    retrieval_audit = []

    for plan in subquery_plans:
        sq = plan["subquery"]
        k = int(plan["retrieval_k"])

        chunks, latency = hybrid_retrieve(sq, k)
        total_latency += latency

        retrieval_audit.append({
            "subquery": sq,
            "difficulty": plan["difficulty"],
            "retrieval_k": k,
            "returned_chunks": len(chunks),
        })

        for c in chunks:
            key = c["faiss_id"]
            prev = all_chunks.get(key)
            if prev is None or c["rerank_score"] > prev["rerank_score"]:
                kept = dict(c)
                kept["matched_subqueries"] = [sq]
                kept["matched_subquery_difficulties"] = [plan["difficulty"]]
                all_chunks[key] = kept
            else:
                prev.setdefault("matched_subqueries", []).append(sq)
                prev.setdefault("matched_subquery_difficulties", []).append(
                    plan["difficulty"]
                )

    merged = list(all_chunks.values())

    if not merged:
        return [], total_latency, retrieval_audit

    # Single decomposed question:
    # preserve exactly its own adaptive retrieval depth.
    if len(subquery_plans) == 1:
        final_k = int(subquery_plans[0]["retrieval_k"])
    else:
        # Multiple decomposed questions:
        # respect the sum of their requested evidence budgets, while applying
        # one global cap to avoid runaway context growth.
        final_k = min(
            DECOMPOSED_FINAL_CONTEXT_CAP,
            sum(int(p["retrieval_k"]) for p in subquery_plans),
            len(merged),
        )

    # Globally rerank the merged, deduplicated evidence against the ORIGINAL
    # user question to create one coherent final context.
    fused_like = [
        {
            "faiss_id": c["faiss_id"],
            "rrf_score": c.get("rrf_score", 0.0),
            "dense_rank": c.get("dense_rank"),
            "bm25_rank": c.get("bm25_rank"),
            "dense_score": c.get("dense_score"),
            "bm25_score": c.get("bm25_score"),
        }
        for c in merged
    ]

    final_chunks = rerank_candidates(
        original_query,
        fused_like,
        top_k=final_k,
    )

    # Restore audit metadata after global reranking.
    merged_lookup = {c["faiss_id"]: c for c in merged}
    for c in final_chunks:
        old = merged_lookup.get(c["faiss_id"], {})
        c["matched_subqueries"] = old.get("matched_subqueries", [])
        c["matched_subquery_difficulties"] = old.get(
            "matched_subquery_difficulties", []
        )

    return final_chunks, total_latency, retrieval_audit


def generate_rag_answer(
    query: str,
    final_k: int,
    subqueries: Optional[list[str]] = None,
):
    """
    Normal Medical QA path.
    Uses the single query's adaptive final_k exactly as before.
    """
    subqueries = subqueries or [query]

    # Normal QA should normally contain one query. If more than one is supplied,
    # retrieve each using the same final_k, then globally rerank.
    plans = [
        {
            "subquery": sq,
            "difficulty": None,
            "retrieval_k": int(final_k),
            "short_reason": "Normal-QA retrieval budget.",
        }
        for sq in subqueries
    ]

    chunks, retrieval_latency, retrieval_audit = _retrieve_planned_subqueries(
        query,
        plans,
    )
    context = chunks_to_context(chunks)

    answer, gen_meta = vllm_chat(
        [
            {"role": "system", "content": RAG_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"User question:\n{query}\n\n"
                    f"Retrieved evidence:\n{context}\n\n"
                    "Answer the user's question."
                ),
            },
        ],
        max_tokens=GENERATOR_MAX_TOKENS,
    )

    return {
        "answer": answer,
        "retrieved_chunks": chunks,
        "retrieved_context": context,
        "retrieval_latency_s": float(retrieval_latency),
        "retrieval_audit": retrieval_audit,
        "generation_meta": gen_meta,
    }


def generate_decomposed_rag_answer(
    query: str,
    subquery_plans: list[dict],
):
    """
    Clarified AMBIGUOUS / CONTRADICTION and OLD_RELEVANT path.

    Every decomposed subquery independently reuses the same Medical-QA
    adaptive retrieval policy:
      easy -> 3-4 chunks
      mid  -> 5-8 chunks
      hard -> 9-15 chunks

    Evidence is merged, deduplicated, globally reranked, then ONE final answer
    is generated for the original user query.
    """
    chunks, retrieval_latency, retrieval_audit = _retrieve_planned_subqueries(
        query,
        subquery_plans,
    )
    context = chunks_to_context(chunks)

    plan_text = "\n".join(
        f"- {p['subquery']} | difficulty={p['difficulty']} | top_k={p['retrieval_k']}"
        for p in subquery_plans
    )

    answer, gen_meta = vllm_chat(
        [
            {"role": "system", "content": RAG_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Original user question:\n{query}\n\n"
                    f"Decomposed retrieval plan:\n{plan_text}\n\n"
                    f"Merged and reranked evidence:\n{context}\n\n"
                    "Synthesize one answer to the ORIGINAL user question."
                ),
            },
        ],
        max_tokens=GENERATOR_MAX_TOKENS,
    )

    return {
        "answer": answer,
        "retrieved_chunks": chunks,
        "retrieved_context": context,
        "retrieval_latency_s": float(retrieval_latency),
        "retrieval_audit": retrieval_audit,
        "generation_meta": gen_meta,
    }


## Updated-information web agent: Tavily + trusted-source scraper

In [ ]:
from getpass import getpass
from urllib.parse import urlparse

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY", "").strip()

def ensure_tavily_key():
    global TAVILY_API_KEY
    if not TAVILY_API_KEY:
        TAVILY_API_KEY = getpass(
            "Enter TAVILY_API_KEY (input hidden; leave empty to disable web route): "
        ).strip()
    return bool(TAVILY_API_KEY)


def domain_is_trusted(url: str) -> bool:
    try:
        host = urlparse(url).hostname or ""
        host = host.lower()
        return any(
            host == d or host.endswith("." + d)
            for d in TRUSTED_MEDICAL_DOMAINS
        )
    except Exception:
        return False


def scrape_webpage(url: str, max_chars: int = 12000) -> str:
    """Simple text scraper used only after a trusted-domain Tavily result."""
    try:
        resp = requests.get(
            url,
            timeout=15,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 MedicalRAGResearch/1.0"
                )
            },
        )
        resp.raise_for_status()
        ctype = resp.headers.get("content-type", "").lower()
        if "text/html" not in ctype:
            return ""
        soup = BeautifulSoup(resp.text, "html.parser")
        for tag in soup(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        text = " ".join(soup.stripped_strings)
        return text[:max_chars]
    except Exception:
        return ""


def tavily_search_and_scrape(query: str):
    if not ensure_tavily_key():
        raise RuntimeError(
            "Tavily key not provided. Current-information route cannot run."
        )

    from tavily import TavilyClient
    client = TavilyClient(api_key=TAVILY_API_KEY)

    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=TAVILY_MAX_RESULTS,
        include_domains=TRUSTED_MEDICAL_DOMAINS,
        topic="general",
    )

    results = []
    for item in response.get("results", []):
        url = str(item.get("url") or "")
        if not url or not domain_is_trusted(url):
            continue
        results.append({
            "title": item.get("title"),
            "url": url,
            "snippet": item.get("content") or "",
        })

    for item in results[:WEB_SCRAPE_TOP_N]:
        item["scraped_text"] = scrape_webpage(item["url"])

    return results


WEB_SYSTEM = r"""
You are an evidence-grounded medical information assistant using current trusted web sources.

Rules:
- Answer using only the supplied current-source material.
- Cite web sources inline as [W1], [W2], etc.
- Do not invent URLs or citations.
- State uncertainty if current evidence is incomplete.
- Use professional medical language without claiming to be a physician.
- If urgent medical attention is indicated, state that clearly.
"""


def web_generate_answer(query: str):
    t0 = time.perf_counter()
    results = tavily_search_and_scrape(query)
    search_latency = time.perf_counter() - t0

    if not results:
        return {
            "answer": (
                "I could not obtain current information from the configured "
                "trusted medical sources, so I cannot safely answer this as a "
                "current-information question."
            ),
            "web_sources": [],
            "web_context": "",
            "web_latency_s": float(search_latency),
            "generation_meta": {},
        }

    blocks = []
    for i, r in enumerate(results, 1):
        body = r.get("scraped_text") or r.get("snippet") or ""
        blocks.append(
            f"[W{i}]\nTitle: {r.get('title')}\nURL: {r.get('url')}\n"
            f"Evidence:\n{body}"
        )
    context = "\n\n".join(blocks)

    answer, gen_meta = vllm_chat(
        [
            {"role": "system", "content": WEB_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Question:\n{query}\n\n"
                    f"Current trusted-source evidence:\n{context}"
                ),
            },
        ],
        max_tokens=GENERATOR_MAX_TOKENS,
    )

    return {
        "answer": answer,
        "web_sources": results,
        "web_context": context,
        "web_latency_s": float(search_latency),
        "generation_meta": gen_meta,
    }

## Post-generation professionalism, privacy, and groundedness checker

In [ ]:
POSTCHECK_SYSTEM = r"""
You are a strict quality-control evaluator for a medical RAG answer.

Evaluate:
1) professionalism_ok:
   Professional clinical communication; clear, respectful, appropriately cautious;
   does not pretend to be the user's physician.

2) third_party_privacy_leak:
   True only if the answer reveals or infers another person's private medical
   information inappropriately. The user's own health information is not
   automatically a third-party privacy leak.

3) groundedness:
   A number from 0.0 to 1.0 representing how well the answer's factual medical
   claims are supported by the supplied evidence.

4) cited_ids_valid:
   Every [C#] or [W#] citation appearing in the answer exists in the supplied
   valid ID list.

Return ONLY JSON:
{
  "professionalism_ok": true,
  "third_party_privacy_leak": false,
  "groundedness": 0.0,
  "cited_ids_valid": true,
  "unsupported_claims": ["..."],
  "quality_note": "short audit note"
}
"""

REPAIR_SYSTEM = r"""
Rewrite the medical answer so it:
- remains concise and professional,
- contains no third-party private health information,
- makes only evidence-supported medical claims,
- cites only the valid evidence IDs supplied,
- states insufficiency/uncertainty instead of guessing,
- preserves appropriate urgent-care advice if supported.

Return only the repaired answer, no commentary.
"""


def postcheck_answer(
    query: str,
    answer: str,
    evidence_context: str,
    valid_ids: list[str],
):
    raw, meta = vllm_chat(
        [
            {"role": "system", "content": POSTCHECK_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Question:\n{query}\n\n"
                    f"Answer:\n{answer}\n\n"
                    f"Valid evidence IDs: {valid_ids}\n\n"
                    f"Evidence:\n{evidence_context}"
                ),
            },
        ],
        max_tokens=400,
    )
    obj = extract_json_object(raw)
    result = {
        "professionalism_ok": bool(obj.get("professionalism_ok", False)),
        "third_party_privacy_leak": bool(
            obj.get("third_party_privacy_leak", False)
        ),
        "groundedness": float(obj.get("groundedness", 0.0)),
        "cited_ids_valid": bool(obj.get("cited_ids_valid", False)),
        "unsupported_claims": obj.get("unsupported_claims") or [],
        "quality_note": str(obj.get("quality_note") or ""),
    }
    return result, meta


def maybe_repair_answer(
    query: str,
    answer: str,
    evidence_context: str,
    valid_ids: list[str],
    check: dict,
):
    fail = (
        not check["professionalism_ok"]
        or check["third_party_privacy_leak"]
        or not check["cited_ids_valid"]
        or check["groundedness"] < GROUNDEDNESS_REPAIR_THRESHOLD
    )
    if not AUTO_REPAIR or not fail:
        return answer, None

    repaired, meta = vllm_chat(
        [
            {"role": "system", "content": REPAIR_SYSTEM},
            {
                "role": "user",
                "content": (
                    f"Question:\n{query}\n\n"
                    f"Original answer:\n{answer}\n\n"
                    f"Valid evidence IDs: {valid_ids}\n\n"
                    f"Evidence:\n{evidence_context}"
                ),
            },
        ],
        max_tokens=GENERATOR_MAX_TOKENS,
    )
    return repaired, meta

## LangGraph state and nodes

In [ ]:
from langgraph.graph import StateGraph, START, END

class MedicalRAGState(TypedDict, total=False):
    query: str
    clarification_context: str

    category: str
    difficulty: Optional[str]
    retrieval_k: Optional[int]
    clean_retrieval_query: str
    ignored_stale_context: list[str]
    analyser_reason: str

    needs_clarification: bool
    clarification_question: str

    relationship_summary: str
    retrieval_focus: list[str]
    subqueries: list[str]
    subquery_plans: list[dict]
    subquery_difficulty_meta: list[dict]

    retrieved_chunks: list[dict]
    retrieved_context: str
    web_sources: list[dict]
    web_context: str

    answer: str
    status: str

    postcheck: dict
    pre_repair_answer: str
    repaired: bool

    agent_trace: list[dict]


def append_trace(state, node: str, meta: Optional[dict] = None, **extra):
    trace = list(state.get("agent_trace", []))
    trace.append({
        "node": node,
        **(meta or {}),
        **extra,
    })
    return trace


def node_analyse(state: MedicalRAGState):
    result, meta = analyse_query_text(
        state["query"],
        state.get("clarification_context", ""),
    )
    return {
        "category": result["category"],
        "difficulty": result["difficulty"],
        "retrieval_k": result["retrieval_k"],
        "clean_retrieval_query": result["clean_retrieval_query"],
        "ignored_stale_context": result["ignored_stale_context"],
        "needs_clarification": result["needs_clarification"],
        "clarification_question": result["clarification_question"],
        "analyser_reason": result["short_reason"],
        "agent_trace": append_trace(state, "query_analyser", meta),
    }


def node_clarity(state: MedicalRAGState):
    # This node does not fabricate user clarification.
    question = state.get("clarification_question", "").strip()
    if not question:
        question = (
            "Your query contains ambiguity or conflicting information that could "
            "change the medical answer. Could you clarify the conflicting or "
            "uncertain detail?"
        )
    return {
        "needs_clarification": True,
        "clarification_question": question,
        "answer": question,
        "status": "needs_clarification",
        "agent_trace": append_trace(state, "clearness_agent"),
    }


def node_relationship(state: MedicalRAGState):
    result, meta = relationship_analysis(
        state["query"],
        state.get("clarification_context", ""),
    )
    return {
        "relationship_summary": result["relationship_summary"],
        "retrieval_focus": result["retrieval_focus"],
        "agent_trace": append_trace(state, "relationship_agent", meta),
    }


def node_decompose(state: MedicalRAGState):
    subqueries, decompose_meta = decompose_query(
        state.get("clean_retrieval_query") or state["query"],
        state.get("relationship_summary", ""),
        state.get("clarification_context", ""),
    )

    # IMPORTANT: each decomposed subquery gets its OWN difficulty and adaptive k.
    subquery_plans, difficulty_metas = build_subquery_plans(subqueries)

    trace = append_trace(
        state,
        "decomposition_agent",
        decompose_meta,
        num_subqueries=len(subqueries),
    )
    for plan, meta in zip(subquery_plans, difficulty_metas):
        trace.append({
            "node": "subquery_difficulty_agent",
            **(meta or {}),
            "subquery": plan["subquery"],
            "difficulty": plan["difficulty"],
            "retrieval_k": plan["retrieval_k"],
        })

    return {
        "subqueries": subqueries,
        "subquery_plans": subquery_plans,
        "subquery_difficulty_meta": difficulty_metas,
        "agent_trace": trace,
    }


def node_normal_rag(state: MedicalRAGState):
    difficulty = state.get("difficulty") or "mid"
    k = state.get("retrieval_k") or DEFAULT_K[difficulty]

    result = generate_rag_answer(
        state.get("clean_retrieval_query") or state["query"],
        final_k=k,
        subqueries=[state.get("clean_retrieval_query") or state["query"]],
    )
    return {
        "answer": result["answer"],
        "retrieved_chunks": result["retrieved_chunks"],
        "retrieved_context": result["retrieved_context"],
        "status": "answered",
        "agent_trace": append_trace(
            state,
            "qa_agent",
            result["generation_meta"],
            retrieval_latency_s=result["retrieval_latency_s"],
            final_k=k,
        ),
    }


def node_decomposed_rag(state: MedicalRAGState):
    """
    Reuses the SAME adaptive Medical-QA retrieval policy for every decomposed
    subquery instead of forcing all decomposed cases into 9-15 chunks.
    """
    plans = state.get("subquery_plans") or []

    # Defensive fallback: if decomposition somehow has no plans, classify the
    # original query as one retrieval subquery.
    if not plans:
        fallback_plan, _ = classify_subquery_difficulty(state["query"])
        plans = [fallback_plan]

    result = generate_decomposed_rag_answer(
        state["query"],
        subquery_plans=plans,
    )

    return {
        "answer": result["answer"],
        "retrieved_chunks": result["retrieved_chunks"],
        "retrieved_context": result["retrieved_context"],
        "status": "answered",
        "agent_trace": append_trace(
            state,
            "decomposed_rag_generator",
            result["generation_meta"],
            retrieval_latency_s=result["retrieval_latency_s"],
            subquery_retrieval_audit=result["retrieval_audit"],
            final_context_chunks=len(result["retrieved_chunks"]),
        ),
    }


def node_web(state: MedicalRAGState):
    result = web_generate_answer(state["query"])
    return {
        "answer": result["answer"],
        "web_sources": result["web_sources"],
        "web_context": result["web_context"],
        "status": "answered_from_current_web",
        "agent_trace": append_trace(
            state,
            "updated_info_web_agent",
            result.get("generation_meta") or {},
            web_latency_s=result["web_latency_s"],
        ),
    }


def node_privacy(state: MedicalRAGState):
    return {
        "answer": "I don't have information regarding the query.",
        "status": "blocked_third_party_privacy_request",
        "agent_trace": append_trace(state, "privacy_agent"),
    }


def node_severe_er(state: MedicalRAGState):
    return {
        "answer": (
            "This may require immediate medical attention. Please go to the "
            "nearest emergency department/hospital or call your local emergency "
            "services now. Do not delay care while waiting for an online answer."
        ),
        "status": "emergency_escalation",
        "agent_trace": append_trace(state, "severe_er_agent"),
    }


def node_postcheck(state: MedicalRAGState):
    # Only evidence-generated answers are postchecked here.
    chunks = state.get("retrieved_chunks") or []
    web_sources = state.get("web_sources") or []

    if chunks:
        valid_ids = [c["context_id"] for c in chunks]
        evidence_context = state.get("retrieved_context", "")
    elif web_sources:
        valid_ids = [f"W{i}" for i in range(1, len(web_sources) + 1)]
        evidence_context = state.get("web_context", "")
    else:
        # Fixed privacy/emergency responses do not need evidence-grounded repair.
        return {
            "postcheck": {
                "professionalism_ok": True,
                "third_party_privacy_leak": False,
                "groundedness": None,
                "cited_ids_valid": True,
                "unsupported_claims": [],
                "quality_note": "Fixed safety/privacy route.",
            },
            "repaired": False,
            "agent_trace": append_trace(state, "postcheck_skipped_fixed_route"),
        }

    check, check_meta = postcheck_answer(
        state["query"],
        state["answer"],
        evidence_context,
        valid_ids,
    )

    repaired_answer, repair_meta = maybe_repair_answer(
        state["query"],
        state["answer"],
        evidence_context,
        valid_ids,
        check,
    )

    repaired = repaired_answer != state["answer"]
    trace = append_trace(state, "postcheck_agent", check_meta)
    if repaired:
        trace.append({"node": "repair_agent", **(repair_meta or {})})

    return {
        "pre_repair_answer": state["answer"] if repaired else "",
        "answer": repaired_answer,
        "postcheck": check,
        "repaired": repaired,
        "agent_trace": trace,
    }

##  Build the LangGraph router

In [ ]:
def route_after_analysis(state: MedicalRAGState):
    cat = state["category"]

    # If an ambiguity/contradiction has already been clarified by the user,
    # send it to relationship/decomposition rather than asking again.
    if cat in {"AMBIGUOUS", "CONTRADICTION"}:
        has_clarification = bool(state.get("clarification_context", "").strip())
        still_unclear = bool(state.get("needs_clarification", True))
        if has_clarification and not still_unclear:
            return "relationship"
        return "clarity"

    return {
        "NORMAL": "normal_rag",
        "OLD_RELEVANT": "relationship",
        "UPDATED_NEED": "web",
        "PRIVACY_LEAKAGE": "privacy",
        "SEVERE_ER": "severe_er",
    }.get(cat, "normal_rag")


builder = StateGraph(MedicalRAGState)

builder.add_node("analyse", node_analyse)
builder.add_node("clarity", node_clarity)
builder.add_node("relationship", node_relationship)
builder.add_node("decompose", node_decompose)
builder.add_node("normal_rag", node_normal_rag)
builder.add_node("decomposed_rag", node_decomposed_rag)
builder.add_node("web", node_web)
builder.add_node("privacy", node_privacy)
builder.add_node("severe_er", node_severe_er)
builder.add_node("postcheck", node_postcheck)

builder.add_edge(START, "analyse")

builder.add_conditional_edges(
    "analyse",
    route_after_analysis,
    {
        "normal_rag": "normal_rag",
        "clarity": "clarity",
        "relationship": "relationship",
        "web": "web",
        "privacy": "privacy",
        "severe_er": "severe_er",
    },
)

builder.add_edge("clarity", END)
builder.add_edge("relationship", "decompose")
builder.add_edge("decompose", "decomposed_rag")

builder.add_edge("normal_rag", "postcheck")
builder.add_edge("decomposed_rag", "postcheck")
builder.add_edge("web", "postcheck")
builder.add_edge("privacy", "postcheck")
builder.add_edge("severe_er", "postcheck")
builder.add_edge("postcheck", END)

medical_rag_graph = builder.compile()
print("LangGraph compiled.")

## Single-query runner

`process_query()` is suitable for API/batch usage.

Important: for an ambiguous/contradictory query, batch mode **does not invent a clarification**. It returns `status="needs_clarification"` plus the clarification question.

In [ ]:
def process_query(
    query: str,
    clarification_context: str = "",
):
    initial: MedicalRAGState = {
        "query": str(query).strip(),
        "clarification_context": str(clarification_context).strip(),
        "agent_trace": [],
    }
    result = medical_rag_graph.invoke(initial)
    return result


# Example smoke test after all models are loaded:
# result = process_query("What are common symptoms of iron deficiency anemia?")
# print(result["category"], result.get("difficulty"), result["answer"])

## Interactive clarification loop — maximum 10 user turns

In [ ]:
def interactive_medical_query(first_query: str):
    """
    Real interactive path for AMBIGUOUS / CONTRADICTION.
    The system asks the actual user for clarification; it never self-fabricates it.
    """
    original_query = str(first_query).strip()
    clarification_history = []

    for turn in range(MAX_CLARIFICATION_TURNS + 1):
        clarification_context = "\n".join(clarification_history)
        result = process_query(
            original_query,
            clarification_context=clarification_context,
        )

        if result.get("status") != "needs_clarification":
            print("\nFinal category:", result.get("category"))
            print("Answer:\n", result.get("answer"))
            return result

        if turn >= MAX_CLARIFICATION_TURNS:
            final = dict(result)
            final["status"] = "clarification_failed"
            final["answer"] = (
                "The query still contains unresolved ambiguity or contradictory "
                "information. Please restate it clearly and try again."
            )
            print(final["answer"])
            return final

        print(
            f"\nClarification {turn + 1}/{MAX_CLARIFICATION_TURNS}: "
            f"{result['clarification_question']}"
        )
        user_reply = input("Your clarification: ").strip()
        if not user_reply:
            user_reply = "[User did not provide clarification.]"

        clarification_history.append(
            f"Clarification question: {result['clarification_question']}\n"
            f"User reply: {user_reply}"
        )

    raise RuntimeError("Unexpected clarification-loop termination.")

## Load benchmark from Drive/local storage

In [ ]:
benchmark_path = next((p for p in BENCHMARK_CANDIDATES if p.exists()), None)

if benchmark_path is None:
    print("No benchmark found automatically.")
    print("Expected one of:")
    for p in BENCHMARK_CANDIDATES:
        print(" -", p)
    print(
        "\nSet benchmark_path manually, for example:\n"
        "benchmark_path = Path('/content/drive/MyDrive/your_benchmark.xlsx')"
    )
else:
    print("Using benchmark:", benchmark_path)

In [ ]:
def load_benchmark(path: str | Path):
    path = Path(path)
    df = pd.read_excel(path, sheet_name="Benchmark")
    if "Question" not in df.columns:
        raise ValueError("Benchmark sheet must contain a 'Question' column.")
    if "ID" not in df.columns:
        df["ID"] = [f"Q{i+1:04d}" for i in range(len(df))]
    df = df[df["Question"].notna()].copy()
    df["ID"] = df["ID"].astype(str)
    df["Question"] = df["Question"].astype(str)
    return df.reset_index(drop=True)


if benchmark_path is not None:
    benchmark_df = load_benchmark(benchmark_path)
    print("Questions:", len(benchmark_df))
    display(benchmark_df.head())

## Batch generation + Drive/local checkpointing

For ambiguous/contradictory benchmark rows there is no live user to clarify them. The batch runner therefore saves:
- the predicted category,
- the clarification question,
- `status = needs_clarification`,
- and does **not** fabricate a medical answer.

That behavior is important for a valid benchmark of the router.

In [ ]:
def _jsonable(value):
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False)
    return value


def flatten_result_for_table(base_row: dict, result: dict):
    chunks = result.get("retrieved_chunks") or []
    web_sources = result.get("web_sources") or []

    return {
        **base_row,
        "predicted_category": result.get("category"),
        "predicted_difficulty": result.get("difficulty"),
        "retrieval_k": result.get("retrieval_k"),
        "status": result.get("status"),
        "analyser_reason": result.get("analyser_reason"),
        "ignored_stale_context": _jsonable(
            result.get("ignored_stale_context") or []
        ),
        "clarification_question": result.get("clarification_question"),
        "relationship_summary": result.get("relationship_summary"),
        "subqueries": _jsonable(result.get("subqueries") or []),
        "subquery_plans": _jsonable(result.get("subquery_plans") or []),
        "generated_answer": result.get("answer"),
        "retrieved_chunk_ids": _jsonable(
            [c.get("chunk_id") for c in chunks]
        ),
        "retrieved_context_ids": _jsonable(
            [c.get("context_id") for c in chunks]
        ),
        "web_urls": _jsonable(
            [x.get("url") for x in web_sources]
        ),
        "postcheck": _jsonable(result.get("postcheck") or {}),
        "repaired": bool(result.get("repaired", False)),
        "agent_trace": _jsonable(result.get("agent_trace") or []),
    }


def save_batch_tables(records: list[dict]):
    df = pd.DataFrame(records)
    df.to_csv(BATCH_CSV, index=False)

    # Excel cells cannot safely hold deeply nested objects; they are already JSON strings.
    with pd.ExcelWriter(BATCH_XLSX, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Results")

    return df


def run_benchmark_batch(benchmark_df: pd.DataFrame):
    completed = {}
    records = []

    if RESUME_BATCH and BATCH_JSONL.exists():
        with open(BATCH_JSONL, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                row = json.loads(line)
                completed[str(row["ID"])] = row
        records = list(completed.values())
        print("Loaded previous results:", len(records))

    work_df = benchmark_df
    if MAX_BATCH_ROWS is not None:
        work_df = work_df.head(int(MAX_BATCH_ROWS))

    mode = "a" if RESUME_BATCH else "w"
    with open(BATCH_JSONL, mode, encoding="utf-8") as out:
        for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="Agentic RAG"):
            qid = str(row["ID"])
            if qid in completed:
                continue

            base = row.to_dict()
            try:
                result = process_query(base["Question"])
                flat = flatten_result_for_table(base, result)
                flat["runtime_error"] = ""
            except Exception as e:
                flat = {
                    **base,
                    "predicted_category": None,
                    "predicted_difficulty": None,
                    "retrieval_k": None,
                    "status": "runtime_error",
                    "generated_answer": "",
                    "runtime_error": repr(e),
                }

            out.write(json.dumps(flat, ensure_ascii=False, default=str) + "\n")
            out.flush()
            completed[qid] = flat

            # Periodic tabular checkpoints.
            if len(completed) % 10 == 0:
                save_batch_tables(list(completed.values()))

    final_records = list(completed.values())
    final_df = save_batch_tables(final_records)
    print("Saved JSONL:", BATCH_JSONL)
    print("Saved CSV:", BATCH_CSV)
    print("Saved XLSX:", BATCH_XLSX)
    return final_df

## Audit summary after a batch run

In [ ]:
def show_audit_summary(results_df: pd.DataFrame):
    print("Rows:", len(results_df))

    if "predicted_category" in results_df:
        print("\nPredicted categories:")
        display(
            results_df["predicted_category"]
            .value_counts(dropna=False)
            .rename_axis("category")
            .to_frame("count")
        )

    if "predicted_difficulty" in results_df:
        print("\nNormal-QA difficulty:")
        display(
            results_df["predicted_difficulty"]
            .value_counts(dropna=False)
            .rename_axis("difficulty")
            .to_frame("count")
        )

    if "status" in results_df:
        print("\nStatuses:")
        display(
            results_df["status"]
            .value_counts(dropna=False)
            .rename_axis("status")
            .to_frame("count")
        )